# DDP quickstart: MNIST on CPU with MPI

A compact, natural example of the distributed training path: define the
network, load the data, set up the optimizer, train, and evaluate. The same
code runs unchanged on one rank or many.

The training code is in the cells tagged `ddp-script`. The kernel does not run
them, because real DDP needs `mpiexec` to start one Julia process per rank.
The notebook writes those cells to
`scripts/examples/ddp_mnist_quickstart.jl` and launches the script as a
**2-rank MPI child process**, then displays the rank-0 output. The kernel never
joins a collective.


## Setup

This notebook uses its own environment, `notebook/env`, with Flux dev'ed from
the `../flux-integration` worktree, plus MLDatasets.jl and Plots.jl.


In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, "env"))
Pkg.instantiate()

ENV["GKSwstype"] = "100"                 # headless GR backend
ENV["DATADEPS_ALWAYS_ACCEPT"] = "true"   # accept the one-time MNIST download

using JSON
using MPI
using Plots
using Printf


## 1. Backend

`DistributedUtils.initialize(MPIBackend)` calls `MPI.Init()` on every process.
`backend` answers the questions the rest of the script needs: who am I
(`local_rank`) and how many ranks are there (`total_workers`). Swap
`MPIBackend` for `NCCLBackend` to run on GPUs.


In [ ]:
# 1. Backend: MPI for CPU processes, NCCL for CUDA devices. This example uses CPUs.
using MPI
using Flux
using Flux: DistributedUtils
using Optimisers
using MLUtils
using MLDatasets
using Random, Printf

DistributedUtils.initialize(DistributedUtils.MPIBackend)
backend = DistributedUtils.get_distributed_backend(DistributedUtils.MPIBackend)
rank = DistributedUtils.local_rank(backend)
world = DistributedUtils.total_workers(backend)


## 2. Network

A small MLP. `Flux.flatten` turns each 28x28 image into 784 inputs. The fixed
seed makes the initial parameters identical on every rank, and
`synchronize!!` broadcasts rank 0's copy as a guarantee.


In [ ]:
# 2. Network: a plain MLP. MLDatasets returns 28x28 images; flatten maps them to 784 inputs.
Random.seed!(1234)
model = Chain(
    Flux.flatten,
    Dense(784 => 128, relu),
    Dense(128 => 10),
)

# Every rank must start from identical parameters: rank 0 broadcasts its copy.
model = DistributedUtils.synchronize!!(
    backend, DistributedUtils.FluxDistributedModel(model); root = 0)


## 3. Data

`MNIST` from MLDatasets.jl returns `features` (28x28xN) and integer `targets`;
`onehotbatch` one-hot encodes the labels. `DistributedDataContainer` gives each
rank a disjoint slice of the same dataset, and the `DataLoader` batches it.
Shards are padded cyclically only when the subset size is not divisible by the
rank count; here 12000 and 2000 divide evenly by 2.


In [ ]:
# 3. Data: a deterministic MNIST subset, sharded across ranks.
const TRAIN_N = 12000
const TEST_N = 2000
const BATCH_SIZE = 32
const EPOCHS = 3

function load_split(split, n)
    d = MNIST(split)
    x = d.features[:, :, 1:n]
    y = Flux.onehotbatch(d.targets[1:n], 0:9)
    return x, y
end

x_train, y_train = load_split(:train, TRAIN_N)
x_test, y_test = load_split(:test, TEST_N)

# DistributedDataContainer gives every rank its own slice of the same dataset.
train_data = DistributedUtils.DistributedDataContainer(backend, (x_train, y_train))
test_data = DistributedUtils.DistributedDataContainer(backend, (x_test, y_test))

train_loader = DataLoader(train_data, batchsize = BATCH_SIZE, shuffle = true)
test_loader = DataLoader(test_data, batchsize = BATCH_SIZE, shuffle = false)


## 4. Optimizer

`DistributedOptimizer` wraps an ordinary Optimisers.jl rule. `Flux.setup`
returns a `DistributedOptimizerState`; every `update` inside `Flux.train!`
averages the gradients across ranks first, then applies Adam locally. All
communication stays outside automatic differentiation.


In [ ]:
# 4. Optimizer: wrap Adam in DistributedOptimizer. Flux.setup returns a
# DistributedOptimizerState that averages gradients once per update, outside AD.
opt_rule = DistributedUtils.DistributedOptimizer(backend, Optimisers.Adam(1.0f-3))
opt_state = Flux.setup(opt_rule, model)


## 5. Train and evaluate

`Flux.train!` consumes the sharded loader and updates through the wrapper
state. Evaluation runs on the local test shard and then sums loss, correct
counts, and totals across ranks with one `allreduce!`. Only rank 0 writes the
shared metrics file.


In [ ]:
# 5. Evaluation: loss and accuracy over the local shard, summed across ranks.
function evaluate(model, loader, backend)
    correct = 0
    total = 0
    loss_sum = 0.0
    for (x, y) in loader
        y_hat = model(x)
        loss_sum += sum(Flux.Losses.logitcrossentropy(y_hat, y; agg = identity))
        correct += sum(Flux.onecold(y_hat) .== Flux.onecold(y))
        total += size(y, 2)
    end
    buf = Float64[loss_sum, correct, total]
    DistributedUtils.allreduce!(backend, buf, +)
    return (loss = buf[1] / buf[3], accuracy = buf[2] / buf[3])
end


In [ ]:
# 5b. Training loop: Flux.train! updates through the distributed wrapper state.
# Rank 0 alone writes the shared metrics file.
metrics_path = get(ENV, "DDP_QUICKSTART_METRICS", joinpath(pwd(), "ddp_mnist_metrics.tsv"))

if rank == 0
    open(metrics_path, "w") do io
        println(io, "epoch\ttest_loss\ttest_accuracy\telapsed_s")
    end
end

for epoch in 1:EPOCHS
    started = time()
    Flux.train!(model, train_loader, opt_state) do m, x, y
        Flux.Losses.logitcrossentropy(m(x), y)
    end
    m = evaluate(model, test_loader, backend)
    elapsed = time() - started
    if rank == 0
        @printf("epoch %d/%d  test_loss=%.4f  test_accuracy=%.4f  (%.1f s)\n",
            epoch, EPOCHS, m.loss, m.accuracy, elapsed)
        open(metrics_path, "a") do io
            @printf(io, "%d\t%.6f\t%.6f\t%.1f\n", epoch, m.loss, m.accuracy, elapsed)
        end
    end
end


## Run it on two ranks

The next cell extracts the tagged cells into
`scripts/examples/ddp_mnist_quickstart.jl` (so the notebook and the script
cannot drift) and runs it with `MPI.mpiexec()` as a child process. It prints
the exact command, so the same run can be reproduced from a shell.


In [ ]:
# The kernel is a single Julia process; real DDP needs one process per rank
# started by mpiexec. Extract the tagged cells into a script and launch it.
function notebook_script(nb_path; tag = "ddp-script")
    cells = JSON.parsefile(nb_path)["cells"]
    selected = [join(c["source"]) for c in cells
        if c["cell_type"] == "code" &&
           tag in get(get(c, "metadata", Dict{String, Any}()), "tags", String[])]
    header = "# GENERATED from notebook/$(basename(nb_path)) (cells tagged \"$tag\").\n" *
             "# Edit the notebook cells and rerun the notebook to regenerate this file.\n\n"
    return header * join(selected, "\n\n") * "\n"
end

write(SCRIPT, notebook_script(joinpath(@__DIR__, NOTEBOOK)))

mkpath(RUN_DIR)
metrics_path = joinpath(RUN_DIR, "metrics.tsv")
rm(metrics_path; force = true)

julia = joinpath(Sys.BINDIR, "julia")
cmd = setenv(
    `$(MPI.mpiexec()) -n $RANKS $julia --startup-file=no --project=$ENV_DIR $SCRIPT`,
    "DDP_QUICKSTART_METRICS" => metrics_path,
    "DATADEPS_ALWAYS_ACCEPT" => "true",
)
println("run: ", cmd)
println("script: ", SCRIPT)

out = IOBuffer()
err = IOBuffer()
proc = run(pipeline(ignorestatus(cmd), stdout = out, stderr = err))
stdout_text = String(take!(out))
stderr_text = String(take!(err))
println("exit code: ", proc.exitcode)
print(stdout_text)
occursin("Precompiling packages", stdout_text) &&
    println("WARNING: the child recompiled; run Pkg.precompile() in the env first.")
proc.exitcode == 0 || (print(stderr_text); error("2-rank run failed (see output above)"))
isfile(metrics_path) || error("rank 0 did not write $(metrics_path)")


## Results

The table and figure are read back from `temp/ddp_mnist_quickstart/metrics.tsv`,
which rank 0 wrote during training.


In [ ]:
rows = [split(line, '\t') for line in readlines(metrics_path)][2:end]
epochs = parse.(Int, first.(rows))
losses = parse.(Float64, getindex.(rows, 2))
accuracies = parse.(Float64, getindex.(rows, 3))
elapsed = parse.(Float64, getindex.(rows, 4))

println("rank count: ", RANKS)
println("epoch\ttest loss\ttest accuracy\telapsed (s)")
for r in rows
    @printf("%s\t%s\t\t%s\t\t%s\n", r[1], r[2], r[3], r[4])
end

p1 = plot(epochs, losses; marker = :circle, xlabel = "epoch", ylabel = "test loss",
          title = "Test loss", legend = false)
p2 = plot(epochs, accuracies; marker = :circle, xlabel = "epoch", ylabel = "test accuracy",
          title = "Test accuracy", ylim = (0, 1), legend = false)
plot(p1, p2, layout = (1, 2), size = (820, 300))


**Observation.** Test loss falls every epoch and test accuracy rises from
about 0.88 after epoch 1 to about 0.92 after epoch 3 (exact values in the
table). Epoch 1 is slower because it compiles the gradient graph; later epochs
finish in about a second.

**Interpretation.** This is the standard DDP contract: one process per rank,
each rank runs the same code on its own data shard, gradients are averaged
before each optimizer step, and metrics are reduced only for reporting. The
one required addition over single-process training is the
`DistributedOptimizer` wrapper around the rule.


## Reproduce from a shell

The tagged cells are the full script; the notebook regenerates
`scripts/examples/ddp_mnist_quickstart.jl` on every run. Set up the
environment once, then launch with any number of ranks:

```
julia --project=notebook/env -e 'using Pkg; Pkg.instantiate()'
julia --project=notebook/env scripts/examples/ddp_mnist_quickstart.jl            # 1 rank
mpiexecjl -n 2 julia --project=notebook/env scripts/examples/ddp_mnist_quickstart.jl
```

Change `RANKS` in the configuration cell (or `-n` on the command line) to use
more ranks; the script itself needs no edits. If `mpiexecjl` is not installed,
`MPI.install_mpiexecjl()` creates it, and the launch cell above prints the raw
`mpiexec` command it used.

## Scope and limitations

- CPU/MPI only. For GPUs, use `NCCLBackend` and move the model and data to the
  device; see `scripts/examples/train_ddp.jl`.
- Every rank must execute the same collective sequence in the same order. Keep
  the control flow data-independent and the optimizer configuration identical
  on all ranks.
- Shards are equal length. Cyclic padding keeps them equal when the dataset
  size is not divisible by the rank count, but duplicated samples receive
  extra weight in the averaged gradients.
- Finite losses do not imply finite gradients, and rank-dependent
  non-finite values can hang the run. These paths are not demonstrated here;
  see `wiki/audits/002-conditional-wrapper-deferred-behaviors.md`.
- The optimizer state is created identically on every rank by `Flux.setup`
  (Adam starts at zero), so no state synchronization is needed.
